# 01 – EDA: dair-ai/emotion
Goal: understand class balance, text length, duplicates/leakage, and label noise **before** modelling. Write 1–2 sentences under each plot (what you see, what you decide).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from src.data import load_emotion, label_names, MODEL_NAME
ds = load_emotion(); names = label_names(ds); print(ds); print(names)

## 1. Class balance (per split)

In [ ]:
rows = {s: Counter(ds[s]['label']) for s in ds}
df = pd.DataFrame(rows).T[range(6)]; df.columns = names
print(df); print((df.div(df.sum(1), axis=0)*100).round(1))
(df.div(df.sum(1), axis=0)*100).T.plot.bar(figsize=(7,3.5), ylabel='% of split'); plt.tight_layout(); plt.savefig('../results/figures/class_balance.png', dpi=150)

**Write:** which classes are rare (surprise, love)? Why does this make accuracy misleading and why macro-F1 the right metric?

## 2. Text length → justify MAX_LENGTH

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
lens = np.array([len(x) for x in tok(ds['train']['text'])['input_ids']])
print('mean', lens.mean(), 'p95', np.percentile(lens,95), 'p99', np.percentile(lens,99), 'max', lens.max())
plt.figure(figsize=(6,3)); plt.hist(lens, bins=50); plt.xlabel('tokens'); plt.tight_layout(); plt.savefig('../results/figures/token_lengths.png', dpi=150)

**Write:** what fraction is truncated at 64 tokens? (`(lens>64).mean()`)

In [ ]:
print('fraction truncated at 64:', (lens>64).mean())

## 3. Leakage check: duplicate texts within / across splits

In [ ]:
tr, va, te = [set(ds[s]['text']) for s in ('train','validation','test')]
print('dups inside train:', len(ds['train']) - len(tr))
print('train∩val:', len(tr&va), ' train∩test:', len(tr&te), ' val∩test:', len(va&te))
# conflicting labels for the same text?
import collections
d = collections.defaultdict(set)
for t,l in zip(ds['train']['text'], ds['train']['label']): d[t].add(l)
print('texts with >1 label in train:', sum(len(v)>1 for v in d.values()))

**Write:** report the counts. The split is kept official (as required), so if overlap exists, *report it as a limitation* rather than silently deleting test rows.

## 4. Sample tweets per class (feel the label noise)

In [ ]:
for i,n in enumerate(names):
    print('\n==', n)
    for t in ds['train'].filter(lambda x: x['label']==i).shuffle(seed=0)[:4]['text']: print(' -', t)

**Write:** are some labels debatable (love vs joy)? That sets expectations for the error analysis.